In [1]:
import os
from pathlib import Path
from pypdf import PdfReader

# =========================
# Rutas
# =========================


RAW_DIR = Path("data/raw_docs")
DATASET_PRACTICAS = Path("C:\\UNIVERSIDAD\\IDAL\\data\\raw_docs\\DATASET PRACTICAS")


# =========================
# Funciones de lectura
# =========================

def leer_markdown(ruta: Path) -> str:
    """Leo los archivos .md directamente."""
    with open(ruta, "r", encoding="utf-8") as f:
        return f.read()


def leer_pdf(ruta: Path) -> str:
    """Extraigo el texto de un PDF completo."""
    reader = PdfReader(str(ruta))
    texto = []

    for page in reader.pages:
        contenido = page.extract_text()
        if contenido:
            texto.append(contenido)

    return "\n".join(texto)


# =========================
# Cargar documentos
# =========================

def cargar_documentos_practicas():
    """
    Cargo todos los documentos MD y PDF de mi dataset de PRÁCTICAS.
    """
    documentos = []

    if not DATASET_PRACTICAS.exists():
        raise FileNotFoundError(f"No existe la carpeta: {DATASET_PRACTICAS}")

    print(f"📂 Cargando documentos desde: {DATASET_PRACTICAS}")

    for archivo in DATASET_PRACTICAS.rglob("*"):

        if archivo.suffix.lower() == ".md":
            texto = leer_markdown(archivo)
            tipo = "md"

        elif archivo.suffix.lower() == ".pdf":
            texto = leer_pdf(archivo)
            tipo = "pdf"

        else:
            continue  # ignorar otros formatos

        documentos.append({
            "archivo": archivo.name,
            "ruta": str(archivo),
            "dominio": "practicas",   # dominio fijo
            "tipo": tipo,
            "texto": texto
        })

    return documentos


# =========================
# Main
# =========================

if __name__ == "__main__":
    documentos = cargar_documentos_practicas()

    print(f"\nDocumentos cargados: {len(documentos)}")

    for doc in documentos[:5]:
        print("-" * 50)
        print("Archivo:", doc["archivo"])
        print("Dominio:", doc["dominio"])
        print("Tipo:", doc["tipo"])
        print("Texto inicial:")
        print(doc["texto"][:500])


In [ ]:
from pathlib import Path
import json
import re
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# =========================
# Rutas
# =========================

RAW_DIR = Path("C:\\UNIVERSIDAD\\IDAL\\data\\raw_docs\\DATASET PRACTICAS")
OUTPUT_FILE = Path("data/chunks_semanticos.json")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# =========================
# Lectura de documentos
# =========================

def leer_markdown(ruta):
    with open(ruta, "r", encoding="utf-8") as f:
        return f.read()


def leer_pdf(ruta):
    reader = PdfReader(str(ruta))
    paginas = []

    for i, page in enumerate(reader.pages):
        texto = page.extract_text() or ""
        paginas.append({
            "pagina": i + 1,
            "texto": texto
        })

    return paginas

# =========================
# Detección de títulos
# =========================

def limpiar_titulo(linea):
    linea = linea.strip()
    linea = re.sub(r"^#{1,6}\s*", "", linea)
    linea = re.sub(r"^[\*\-\–•]\s*", "", linea)
    linea = linea.replace("**", "").replace("__", "").replace("`", "")
    return linea.strip()


def es_titulo(linea):
    linea = linea.strip()

    if not linea:
        return False

    if re.fullmatch(r"[-_*]{3,}", linea):
        return False

    # Títulos Markdown reales
    if re.match(r"^#{1,6}\s+", linea):
        return True

    # Títulos en negrita tipo "**Título**"
    if re.match(r"^\*\*[^*]{3,120}\*\*:?$", linea):
        return True

    # Títulos tipo "4.1 Algo"
    if re.match(r"^\d+[\.\)]\s+[A-ZÁÉÍÓÚÜÑ]", linea):
        return True

    # Artículos legales
    if re.match(r"^(Artículo|Articulo|ARTÍCULO|ARTICLE|Article)\s+\d+", linea):
        return True

    # Títulos en mayúsculas
    if re.match(r"^([IVXLCDM]+\.\s+)?[A-ZÁÉÍÓÚÜÑ0-9\s,/:()-]{5,140}$", linea):
        return True

    return False



def dividir_por_titulos(texto):
    lineas = texto.splitlines()

    secciones = []
    titulo_actual = "Inicio del documento"
    contenido_actual = []

    for linea in lineas:
        if es_titulo(linea):
            if contenido_actual:
                secciones.append({
                    "titulo": titulo_actual,
                    "texto": "\n".join(contenido_actual).strip()
                })

            titulo_actual = limpiar_titulo(linea)
            contenido_actual = [linea]
        else:
            contenido_actual.append(linea)

    if contenido_actual:
        secciones.append({
            "titulo": titulo_actual,
            "texto": "\n".join(contenido_actual).strip()
        })

    return secciones

# =========================
# Chunking semántico
# =========================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1400,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", "; ", ", ", " ", ""]
)

todos_los_chunks = []

for archivo in RAW_DIR.rglob("*"):
    if archivo.suffix.lower() not in [".md", ".pdf"]:
        continue

    if archivo.suffix.lower() == ".md":
        texto = leer_markdown(archivo)
        secciones = dividir_por_titulos(texto)

        for seccion_id, seccion in enumerate(secciones):
            if not seccion["texto"].strip():
                continue

            subchunks = splitter.split_text(seccion["texto"])

            for subchunk_id, subchunk in enumerate(subchunks):
                if len(subchunk.strip()) < 50:
                    continue

                todos_los_chunks.append({
                    "archivo": archivo.name,
                    "ruta": str(archivo),
                    "tipo": "markdown",
                    "pagina": None,
                    "dominio": "practicas",
                    "seccion": seccion["titulo"],
                    "seccion_id": seccion_id,
                    "subchunk_id": subchunk_id,
                    "texto": subchunk
                })

    elif archivo.suffix.lower() == ".pdf":
        paginas = leer_pdf(archivo)

        for pagina in paginas:
            secciones = dividir_por_titulos(pagina["texto"])

            for seccion_id, seccion in enumerate(secciones):
                if not seccion["texto"].strip():
                    continue

                subchunks = splitter.split_text(seccion["texto"])

                for subchunk_id, subchunk in enumerate(subchunks):
                    if len(subchunk.strip()) < 50:
                        continue

                    todos_los_chunks.append({
                        "archivo": archivo.name,
                        "ruta": str(archivo),
                        "tipo": "pdf",
                        "pagina": pagina["pagina"],
                        "dominio": "practicas",
                        "seccion": seccion["titulo"],
                        "seccion_id": seccion_id,
                        "subchunk_id": subchunk_id,
                        "texto": subchunk
                    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(todos_los_chunks, f, ensure_ascii=False, indent=2)

print(f"Chunks creados: {len(todos_los_chunks)}")
print(f"Guardados en: {OUTPUT_FILE}")


In [4]:
import json
import re
from pathlib import Path

INPUT_FILE = Path("data/chunks_semanticos.json")
OUTPUT_FILE = Path("data/chunks_filtrados.json")

MIN_PALABRAS_MD = 15
MIN_PALABRAS_PDF = 40


def limpiar_texto(texto):
    texto = texto.strip()
    texto = re.sub(r"\s+", " ", texto)
    return texto


def contar_palabras(texto):
    return len(texto.split())


def es_chunk_relevante(chunk):
    texto = limpiar_texto(chunk.get("texto", ""))

    tipo = chunk.get("tipo", "").lower()

    # =========================
    # 1) Filtro por mínimo de palabras
    # =========================
    if tipo == "pdf":
        min_palabras = MIN_PALABRAS_PDF
    else:
        min_palabras = MIN_PALABRAS_MD

    if contar_palabras(texto) < min_palabras:
        return False

    # =========================
    # 2) Filtro por proporción de letras
    # =========================
    letras = re.findall(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]", texto)
    proporcion_letras = len(letras) / max(len(texto), 1)

    if proporcion_letras < 0.45:
        return False

    # =========================
    # 3) Filtro de basura general
    # =========================
    patrones_basura = [
        r"^página\s+\d+",
        r"^page\s+\d+",
        r"^\d+$",
        r"^índice$",
        r"^tabla de contenidos$",
        r"^www\.",
        r"^https?://",
    ]

    texto_lower = texto.lower()

    for patron in patrones_basura:
        if re.search(patron, texto_lower):
            return False

    # =========================
    # 4) Filtro ESPECÍFICO para eliminar ÍNDICES
    # =========================

    # A) Si contiene MUCHOS números → índice
    numeros = re.findall(r"\b\d{1,3}\b", texto)
    if len(numeros) >= 10:
        return False

    # B) Si contiene palabras típicas de índice
    if re.search(r"\b(índice|contents|table of contents|sumario)\b", texto_lower):
        return False

    # C) Si muchas líneas terminan en número → índice
    lineas = texto.split("\n")
    terminan_en_numero = sum(1 for l in lineas if re.search(r"\d+$", l.strip()))
    if terminan_en_numero >= 4:
        return False

    # D) Si tiene estructura "título número título número..." → índice
    if len(numeros) > 5 and re.search(r"[A-Za-zÁÉÍÓÚÜÑ].+\d{1,3}", texto):
        return False

    # =========================
    # Si llega aquí → chunk válido
    # =========================
    return True


def main():
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    chunks_filtrados = []

    eliminados_md = 0
    eliminados_pdf = 0

    for chunk in chunks:
        if es_chunk_relevante(chunk):
            chunk["texto"] = limpiar_texto(chunk["texto"])
            chunks_filtrados.append(chunk)
        else:
            if chunk.get("tipo", "").lower() == "pdf":
                eliminados_pdf += 1
            else:
                eliminados_md += 1

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(chunks_filtrados, f, ensure_ascii=False, indent=2)

    print(f"Chunks originales: {len(chunks)}")
    print(f"Chunks filtrados: {len(chunks_filtrados)}")
    print(f"Chunks eliminados: {len(chunks) - len(chunks_filtrados)}")
    print(f"Eliminados MD: {eliminados_md}")
    print(f"Eliminados PDF: {eliminados_pdf}")
    print(f"Guardado en: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


In [7]:
import json
from pathlib import Path

def comprobar_top_5_mas_cortos(ruta_json="data/chunks_filtrados.json"):
    """
    Cargo el JSON final de prácticas que he generado, ordeno los chunks por longitud de su 
    campo 'texto' y saco los 5 primeros. Así audito que mi limpieza ha ido perfecta.
    """
    ruta = Path(ruta_json)
    if not ruta.exists():
        print(f"❌ Error: No se encuentra el archivo en '{ruta_json}'.")
        print("Asegúrate de haber ejecutado completamente las celdas anteriores del pipeline.")
        return

    # 1. Cargar el archivo generado
    with open(ruta, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    if not chunks:
        print("⚠️ El archivo JSON existe pero está completamente vacío.")
        return

    # 2. Mapear y extraer las longitudes reales del campo 'texto'
    chunks_con_medida = []
    for idx, chunk in enumerate(chunks):
        chunks_con_medida.append({
            "indice_original": idx,
            "archivo": chunk.get("archivo", "Desconocido"),
            "seccion": chunk.get("seccion", "Sin sección"),
            "texto": chunk.get("texto", ""),
            "longitud": len(chunk.get("texto", ""))
        })

    # 3. Ordenar de menor a mayor por la clave 'longitud'
    chunks_ordenados = sorted(chunks_con_medida, key=lambda x: x["longitud"])

    # 4. Imprimir el reporte analítico por pantalla
    print("=" * 80)
    print("🔬 AUDITORÍA: TOP 5 CHUNKS MÁS CORTOS DEL DATASET DE PRÁCTICAS")
    print("=" * 80)
    print(f"📊 Total de chunks analizados en el JSON: {len(chunks)}")
    print("Objetivo: Verificar que los filtros hayan purgado índices, links puros y títulos vacíos.")
    print("-" * 80)

    # Tomamos únicamente los 5 primeros (los más pequeños)
    for i, item in enumerate(chunks_ordenados[:5], start=1):
        print(f"🥇 Puesto #{i} | Longitud: {item['longitud']} caracteres | Índice en JSON: {item['indice_original']}")
        print(f"📄 Archivo Origen: {item['archivo']}")
        print(f"🗺️ Ruta Jerárquica: {item['seccion']}")
        print(f"📝 Contenido extraído:")
        
        # Reemplazamos los saltos de línea físicos por '\n' visuales para que el print sea compacto
        texto_limpio_visual = item['texto'].replace('\n', ' \\n ')
        print(f"   ➔ \"{texto_limpio_visual}\"")
        print("-" * 80)

# ==========================================
# EJECUTAR COMPROBACIÓN
# ==========================================
if __name__ == "__main__":
    comprobar_top_5_mas_cortos()